In [1]:
library(data.table)
library(dplyr)
library(tidyverse)


Attaching package: 'dplyr'


The following objects are masked from 'package:data.table':

    between, first, last


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


-- Attaching packages ------------------------------------------------------------------- tidyverse 1.3.1 --

v ggplot2 3.5.1     v purrr   0.3.4
v tibble  3.1.3     v stringr 1.4.0
v tidyr   1.1.3     v forcats 0.5.1
v readr   2.0.0     

-- Conflicts ---------------------------------------------------------------------- tidyverse_conflicts() --
x dplyr::between()   masks data.table::between()
x dplyr::filter()    masks stats::filter()
x dplyr::first()     masks data.table::first()
x dplyr::lag()       masks stats::lag()
x dplyr::last()      masks data.table::last()
x purrr::transpose() masks data.table::transpose()



## 2. Set list file
This file lists variants within each set/gene to use when building masks. Each line contains the set/gene name followed by a chromosome and physical position for the set/gene, then by a comma-separated list of variants included in the set/gene.

A1BG 19  58346922  19:58346922:C:A,19:58346924:G:A,...

A1CF 10  50806630  10:50806630:A:G,10:50806630:A:AT,...




In [ ]:
setwd("/medpop/esp2/projects/MG")

In [5]:
d <- fread("chr1.MGB_53K_WES.vep.hc_LOF.annotation", col.names=c("VAR","GENE","ANNOT"))

head(d)

VAR,GENE,ANNOT
<chr>,<chr>,<chr>
chr1:931090:G:C,ENSG00000187634,LoF
chr1:935842:C:T,ENSG00000187634,LoF
chr1:935898:T:A,ENSG00000187634,LoF
chr1:939111:C:T,ENSG00000187634,LoF
chr1:944705:CTG:C,ENSG00000188976,LoF
chr1:944706:TG:T,ENSG00000188976,LoF


In [6]:
table(table(d$VAR))


    1 
16923 

In [14]:
library(data.table)

for(i in 1:22){

    anno <- fread(
        paste0("chr",i,".MGB_53K_WES.vep.hc_LOF.annotation"),
        col.names=c("VAR","GENE","ANNOT")
    )

    anno <- unique(anno, by=c("VAR","GENE"))

    anno[, c("CHR","POS") := tstrsplit(VAR, ":", fixed=TRUE, keep=c(1,2))]
    anno[, POS := as.integer(POS)]

#    setorder(anno, GENE, POS)

    setlist <- anno[
        order(POS),
        .(
            CHR = CHR[1],
            POS = min(POS),
            VARS = paste(VAR, collapse=",")
        ),
        by=GENE
    ]

    fwrite(
        setlist,
        paste0("chr",i,".MGB_53K_WES.vep.hc_LOF.setlist"),
        sep="\t",
        quote=FALSE,
        col.names=FALSE,
        row.names=FALSE
    )

    cat(
        "Chr", i,
        ": ", nrow(setlist),
        " genes, ",
        nrow(anno),
        " variant-gene pairs\n"
    )

    rm(anno, setlist)
    gc()
}

Chr 1 :  1684  genes,  16923  variant-gene pairs
Chr 2 :  1070  genes,  12926  variant-gene pairs
Chr 3 :  891  genes,  10208  variant-gene pairs
Chr 4 :  644  genes,  7548  variant-gene pairs
Chr 5 :  741  genes,  7647  variant-gene pairs
Chr 6 :  858  genes,  8697  variant-gene pairs
Chr 7 :  751  genes,  8202  variant-gene pairs
Chr 8 :  579  genes,  6098  variant-gene pairs
Chr 9 :  618  genes,  6513  variant-gene pairs
Chr 10 :  617  genes,  6767  variant-gene pairs
Chr 11 :  1008  genes,  9979  variant-gene pairs
Chr 12 :  874  genes,  9048  variant-gene pairs
Chr 13 :  275  genes,  3092  variant-gene pairs
Chr 14 :  502  genes,  5094  variant-gene pairs
Chr 15 :  487  genes,  5803  variant-gene pairs
Chr 16 :  669  genes,  7167  variant-gene pairs
Chr 17 :  932  genes,  9697  variant-gene pairs
Chr 18 :  240  genes,  2610  variant-gene pairs
Chr 19 :  1125  genes,  9845  variant-gene pairs
Chr 20 :  439  genes,  3817  variant-gene pairs
Chr 21 :  157  genes,  1710  variant-gene 

In [2]:
library(data.table)

# for(i in 1:22){
i="X"
    anno <- fread(
        paste0("chr",i,".MGB_53K_WES.vep.hc_LOF.annotation"),
        col.names=c("VAR","GENE","ANNOT")
    )

    anno <- unique(anno, by=c("VAR","GENE"))

    anno[, c("CHR","POS") := tstrsplit(VAR, ":", fixed=TRUE, keep=c(1,2))]
    anno[, POS := as.integer(POS)]

#    setorder(anno, GENE, POS)

    setlist <- anno[
        order(POS),
        .(
            CHR = CHR[1],
            POS = min(POS),
            VARS = paste(VAR, collapse=",")
        ),
        by=GENE
    ]

    fwrite(
        setlist,
        paste0("chr",i,".MGB_53K_WES.vep.hc_LOF.setlist"),
        sep="\t",
        quote=FALSE,
        col.names=FALSE,
        row.names=FALSE
    )

    cat(
        "Chr", i,
        ": ", nrow(setlist),
        " genes, ",
        nrow(anno),
        " variant-gene pairs\n"
    )

    rm(anno, setlist)
    gc()
# }

ERROR: Error in fread(paste0("chr", i, ".MGB_53K_WES.vep.hc_LOF.annotation"), : File 'chrX.MGB_53K_WES.vep.hc_LOF.annotation' does not exist or is non-readable. getwd()=='/medpop/esp2/mesbah/tools/meta-gwas-of-clonal-hematopoiesis/02_2.RVAS/mask_files'


## Step 4: Create mask definition file

For pure pLoF burden:

In [ ]:

## cat > lof.mask << EOF
# M1 LoF
# EOF